# Multimodal Beam Prediction with Cross-Modal Knowledge Distillation
### AMBER / DeepSense6G (scenarios 31-34), reduced-epoch reproduction on Kaggle

This trains:
1. A **multimodal Teacher** (camera + LiDAR-BEV + radar + GPS + beam history) fused with a small
   Transformer encoder.
2. A **radar-only Student** distilled from the Teacher (logit KD + a simplified relational-KD term).
3. A **WithoutKD baseline** (same radar-only student, trained with plain focal loss) for comparison,
   mirroring the paper's Table II layout (Teacher / WithoutKD / KD-student).

**Simplifications vs. the original paper** (done deliberately to fit a 6-10 hour Kaggle GPU budget on a
dataset ~2.5x smaller than the paper's CARLA-generated one):
- Only the latest frame of each 5-frame window is used (no temporal stacking).
- Images/LiDAR-BEV/radar are resized to `IMG_SIZE x IMG_SIZE` (default 128, not 256).
- The Teacher uses ResNet-18 (partially frozen) for images and small CNNs for LiDAR/radar, not a
  105M-parameter transformer with ResNet-18/34 branches.
- The relational-KD term uses a single cosine-similarity manifold instead of the paper's
  Euclidean + cosine + hyperbolic three-manifold sum, plus the beam-steering-vector similarity term.

**Before running the full job:** flip `DEBUG = True` a few cells down and do a 2-epoch, 200-sample smoke
test first. Debugging on the full dataset is the most common way people burn through a GPU quota.

This dataset (5,544 train / 1,350 val rows) is meaningfully smaller than the paper's (~13k train rows),
so a full run at the settings below will likely finish well under the 6-10 hour budget - each epoch
prints its own wall time so you can watch progress and raise `TEACHER_EPOCHS` / `STUDENT_EPOCHS` /
`IMG_SIZE` if you want to spend more of the budget.

In [1]:
import os, glob, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected - turn on a GPU accelerator in Kaggle's notebook settings.")

# ---------------- Config you may want to tune ----------------
IMG_SIZE = 128          # spatial size all modalities are resized to (paper used 256; smaller = faster)
BATCH_SIZE = 48
NUM_WORKERS = 4
LATENT_DIM = 128        # shared dimension used for KD comparisons (teacher fused feature / student encoding layer)
COMMON_DIM = 128        # per-modality token dimension fed into the teacher's fusion transformer
NUM_BEAMS = 64

TEACHER_EPOCHS = 25     # paper used 50; this dataset is ~2.5x smaller so fewer epochs are enough
STUDENT_EPOCHS = 25
PATIENCE = 6            # early-stopping patience, in epochs without val-MPR improvement

KD_ALPHA = 0.5          # balance between focal loss and distillation loss (matches the paper's default)
KD_TEMP = 2.0
USE_RELATIONAL = True   # simplified relational-KD term vs. a plain latent-MSE term

DEBUG = False           # set True for a fast smoke test (small subset, 2 epochs) before the full run

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

Device: cuda


## Locate the dataset

Since you uploaded this as a Kaggle Dataset, the original `archive/` wrapper folder from the zip won't
be part of the mounted path - Kaggle mounts your dataset's contents directly under
`/kaggle/input/<your-dataset-slug>/...`. Rather than hardcode that slug, we search for `samples.csv`
under `/kaggle/input` and derive every other path relative to where it's found.

In [2]:
candidates = glob.glob("/kaggle/input/**/samples.csv", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "Could not find samples.csv anywhere under /kaggle/input. "
        "Check that the dataset was attached to this notebook (Add Data, top right)."
    )
SAMPLES_CSV = candidates[0]
print("Found manifest at:", SAMPLES_CSV)

# samples.csv lives at <ROOT>/index/index/samples.csv per the dataset doc,
# so ROOT is three directories up from the file.
DATA_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(SAMPLES_CSV)))
print("Derived DATA_ROOT:", DATA_ROOT)
print("Top-level contents:", os.listdir(DATA_ROOT))

INDEX_DIR = os.path.dirname(SAMPLES_CSV)
BEAM_PWR_PATH = os.path.join(INDEX_DIR, "beam_pwr.npy")
GPS_NORM_PATH = os.path.join(INDEX_DIR, "gps_norm.json")
assert os.path.exists(BEAM_PWR_PATH), BEAM_PWR_PATH
assert os.path.exists(GPS_NORM_PATH), GPS_NORM_PATH

beam_pwr = np.load(BEAM_PWR_PATH)                 # shape (11868, 64)
with open(GPS_NORM_PATH) as f:
    gps_norm = json.load(f)
print("beam_pwr shape:", beam_pwr.shape)


def resolve_path(rel_path, source):
    '''Apply the dataset's documented path-correction rule: the 'data/processed_amber/'
    prefix stored in the manifest is replaced with '<source>/'.'''
    if not isinstance(rel_path, str) or rel_path == "":
        return None
    local = rel_path.replace("data/processed_amber/", source + "/", 1)
    return os.path.join(DATA_ROOT, local)


# gps_norm.json's exact key shape isn't guaranteed (dict vs [min,max] list), so we parse
# defensively and fall back to the values documented in DATASET_CONTEXT.md if parsing fails.
GPS_MINMAX_FALLBACK = {
    "ue_x_1": (-7.620373974330076, 19.01809280985755),
    "ue_y_1": (-2.4172876795492675, 52.311869241120824),
    "ue_x_2": (-7.180620672901669, 18.99639189669229),
    "ue_y_2": (-1.901362025116012, 52.372685988738546),
}

def get_gps_minmax(field):
    try:
        entry = gps_norm[field]
        if isinstance(entry, dict):
            lo = entry.get("min", entry.get("minimum"))
            hi = entry.get("max", entry.get("maximum"))
        else:
            lo, hi = entry[0], entry[1]
        if lo is None or hi is None:
            raise KeyError
        return float(lo), float(hi)
    except Exception:
        return GPS_MINMAX_FALLBACK[field]

Found manifest at: /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset/index/index/samples.csv
Derived DATA_ROOT: /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset
Top-level contents: ['development', 'index', 'adaptation', 'test']
beam_pwr shape: (11868, 64)


## Load and filter the manifest

Per the dataset contract: a row is usable for supervised training only when `beam >= 0` **and**
`pwr_n_nan == 0`. We keep the manifest's own `train`/`val` split rather than reshuffling everything -
the block split already accounts for temporal proximity and session structure.

In [3]:
df = pd.read_csv(SAMPLES_CSV)
print("Total rows:", len(df))
print(df["split"].value_counts())

supervised = df[(df["beam"] >= 0) & (df["pwr_n_nan"] == 0)].copy()
train_df = supervised[supervised["split"] == "train"].reset_index(drop=True)
val_df = supervised[supervised["split"] == "val"].reset_index(drop=True)
print(f"Usable rows -> train: {len(train_df)}  val: {len(val_df)}")

if DEBUG:
    train_df = train_df.sample(min(200, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(min(100, len(val_df)), random_state=SEED).reset_index(drop=True)
    TEACHER_EPOCHS = 2
    STUDENT_EPOCHS = 2
    print(f"[DEBUG MODE] subsetted to train={len(train_df)} val={len(val_df)}, epochs=2")

GPS_COLS = ["ue_x_1", "ue_y_1", "ue_x_2", "ue_y_2"]

Total rows: 11868
split
train               5544
no_target           4191
val                 1350
test                 625
adaptation           100
excluded_nan_pwr      58
Name: count, dtype: int64
Usable rows -> train: 5544  val: 1350


## Dataset class

Loads the **latest frame** (`*_5` columns) of each modality, applies the documented availability masks
(`m_image`, `m_lidar`, `m_radar`, `m_gps`, `m_beam`), and returns a placeholder tensor of the right shape
for anything unavailable rather than skipping the sample.

In [4]:
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_array_resized(path):
    arr = np.load(path).astype(np.float32)        # shape (C, H, W)
    t = torch.from_numpy(arr)
    if t.shape[-2:] != (IMG_SIZE, IMG_SIZE):
        t = F.interpolate(t.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                           mode="bilinear", align_corners=False).squeeze(0)
    return t


def _normalize_one(v, lo, hi):
    if pd.isna(v):
        return 0.0
    return float(np.clip((float(v) - lo) / (hi - lo), 0.0, 1.0))


class BeamDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = resolve_path(row["image_5"], row["source"])
        lidar_path = resolve_path(row["lidar_bev_5"], row["source"])
        radar_path = resolve_path(row["radar_5"], row["source"])

        if int(row["m_image"]) and img_path and os.path.exists(img_path):
            image = IMG_TRANSFORM(Image.open(img_path).convert("RGB"))
        else:
            image = torch.zeros(3, IMG_SIZE, IMG_SIZE)

        if int(row["m_lidar"]) and lidar_path and os.path.exists(lidar_path):
            lidar = load_array_resized(lidar_path)
        else:
            lidar = torch.zeros(1, IMG_SIZE, IMG_SIZE)

        if int(row["m_radar"]) and radar_path and os.path.exists(radar_path):
            radar = load_array_resized(radar_path)
        else:
            radar = torch.zeros(2, IMG_SIZE, IMG_SIZE)

        if int(row["m_gps"]):
            gps = torch.tensor([_normalize_one(row[c], *get_gps_minmax(c)) for c in GPS_COLS],
                                dtype=torch.float32)
        else:
            gps = torch.zeros(len(GPS_COLS))

        beam_hist_idx = []
        for i in range(1, 5):
            v = row[f"beam_hist_{i}"]
            valid = int(row["m_beam"]) == 1 and pd.notna(v) and int(v) >= 0
            beam_hist_idx.append(int(v) + 1 if valid else 0)   # 0 = missing/padding token
        beam_hist = torch.tensor(beam_hist_idx, dtype=torch.long)

        return {
            "image": image, "lidar": lidar, "radar": radar,
            "gps": gps, "beam_hist": beam_hist,
            "label": int(row["beam"]), "pwr_row": int(row["pwr_row"]),
            "idx": idx,   # position within this dataframe - used to align cached teacher outputs
        }


train_ds = BeamDataset(train_df)
val_ds = BeamDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
# Non-shuffled loader used only for caching teacher outputs - order must match train_df row order.
train_loader_ordered = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

print(f"Batches per epoch -> train: {len(train_loader)}  val: {len(val_loader)}")

Batches per epoch -> train: 115  val: 29


## Models

- **Teacher**: image (ResNet-18, partially frozen) + LiDAR-BEV (small CNN) + radar (small CNN) +
  GPS (MLP) + beam history (embedding) -> per-modality tokens -> 2-layer Transformer fusion -> latent
  -> 64-way classifier.
- **Student**: radar CNN -> encoding-layer MLP (this is the KD "latent") -> prediction MLP. Mirrors the
  paper's Fig. 4 radar-only branch.

In [5]:
class ConvEncoder(nn.Module):
    '''Small CNN encoder used for LiDAR-BEV and radar inputs.'''
    def __init__(self, in_channels, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.Conv2d(in_channels, 32, 5, stride=2, padding=2), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(256, out_dim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.proj(h)


class ImageEncoder(nn.Module):
    def __init__(self, out_dim, freeze_backbone=True):
        super().__init__()
        try:
            backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception:
            backbone = torchvision.models.resnet18(pretrained=True)
        backbone.fc = nn.Identity()
        if freeze_backbone:
            for name, p in backbone.named_parameters():
                if not name.startswith("layer4"):
                    p.requires_grad_(False)
        self.backbone = backbone
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class GPSEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 32), nn.ReLU(inplace=True), nn.Linear(32, out_dim))

    def forward(self, x):
        return self.net(x)


class BeamHistoryEncoder(nn.Module):
    def __init__(self, out_dim, vocab_size=65, embed_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.net = nn.Sequential(nn.Linear(embed_dim * 4, 32), nn.ReLU(inplace=True), nn.Linear(32, out_dim))

    def forward(self, x):
        return self.net(self.embed(x).flatten(1))


class TeacherModel(nn.Module):
    def __init__(self, common_dim=COMMON_DIM, latent_dim=LATENT_DIM, num_beams=NUM_BEAMS):
        super().__init__()
        self.image_enc = ImageEncoder(common_dim)
        self.lidar_enc = ConvEncoder(1, common_dim)
        self.radar_enc = ConvEncoder(2, common_dim)
        self.gps_enc = GPSEncoder(common_dim)
        self.hist_enc = BeamHistoryEncoder(common_dim)

        self.modality_embed = nn.Parameter(torch.randn(5, common_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=common_dim, nhead=4, dim_feedforward=common_dim * 2,
            dropout=0.1, batch_first=True,
        )
        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.to_latent = nn.Sequential(nn.Linear(common_dim, latent_dim), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(latent_dim, num_beams)

    def forward(self, batch, return_latent=False):
        tokens = torch.stack([
            self.image_enc(batch["image"]),
            self.lidar_enc(batch["lidar"]),
            self.radar_enc(batch["radar"]),
            self.gps_enc(batch["gps"]),
            self.hist_enc(batch["beam_hist"]),
        ], dim=1)
        tokens = tokens + self.modality_embed.unsqueeze(0)
        fused = self.fusion(tokens).mean(dim=1)
        latent = self.to_latent(fused)
        logits = self.classifier(latent)
        return (logits, latent) if return_latent else logits


class StudentModel(nn.Module):
    '''Radar-only student: radar CNN -> encoding-layer MLP -> prediction MLP (Fig. 4 of the paper).'''
    def __init__(self, latent_dim=LATENT_DIM, num_beams=NUM_BEAMS):
        super().__init__()
        self.radar_enc = ConvEncoder(2, 256)
        self.encoding_layer = nn.Sequential(
            nn.Linear(256, 256), nn.ReLU(inplace=True),
            nn.Linear(256, latent_dim), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(inplace=True),
            nn.Linear(128, num_beams),
        )

    def forward(self, batch, return_latent=False):
        latent = self.encoding_layer(self.radar_enc(batch["radar"]))
        logits = self.classifier(latent)
        return (logits, latent) if return_latent else logits

## Losses

- **Focal loss** on the 64-way beam label (as in the paper).
- **KD (KL-divergence)** between softened teacher/student logits.
- **Relational loss (simplified)**: matches the pairwise cosine-similarity structure between samples in
  a batch, teacher vs. student latents - the core idea of the paper's Relational KD, without the full
  three-manifold (Euclidean + cosine + hyperbolic) computation or the beam-steering-vector term.

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        pt = p.gather(1, target.unsqueeze(1)).squeeze(1)
        logpt = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        return (-((1 - pt) ** self.gamma) * logpt).mean()


def kd_kl_loss(student_logits, teacher_logits, T=KD_TEMP):
    s = F.log_softmax(student_logits / T, dim=-1)
    t = F.softmax(teacher_logits / T, dim=-1)
    return F.kl_div(s, t, reduction="batchmean") * (T ** 2)


def relational_loss(student_latent, teacher_latent):
    s_n = F.normalize(student_latent, dim=-1)
    t_n = F.normalize(teacher_latent, dim=-1)
    return F.mse_loss(s_n @ s_n.t(), t_n @ t_n.t())

## Metrics: Top-K accuracy, mean RSS, mean percentile rank (MPR)

Computed exactly as in the paper (Eqs. 19-23), using `beam_pwr.npy` via each sample's `pwr_row`.

In [7]:
def compute_metrics(all_logits, all_labels, all_pwr_rows, ks=(1, 5, 10)):
    probs = torch.softmax(all_logits, dim=-1)
    topk_acc = {}
    for k in ks:
        topk = probs.topk(k, dim=-1).indices
        topk_acc[f"top{k}"] = (topk == all_labels.unsqueeze(1)).any(dim=1).float().mean().item()

    preds = probs.argmax(dim=-1).numpy()
    pwr_rows = all_pwr_rows.numpy()
    rss_vals, percentiles = [], []
    for pred, prow in zip(preds, pwr_rows):
        row_pwr = beam_pwr[prow]
        rss_vals.append(row_pwr[pred])
        rank = 1 + int((row_pwr > row_pwr[pred]).sum())
        percentiles.append((NUM_BEAMS - rank + 1) / NUM_BEAMS)

    return {**topk_acc, "mean_rss": float(np.mean(rss_vals)), "mpr": float(np.mean(percentiles))}


def move_batch(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels, all_pwr = [], [], []
    for batch in loader:
        labels, pwr_rows = batch["label"], batch["pwr_row"]
        logits = model(move_batch(batch, device))
        all_logits.append(logits.cpu())
        all_labels.append(labels)
        all_pwr.append(pwr_rows)
    return compute_metrics(torch.cat(all_logits), torch.cat(all_labels), torch.cat(all_pwr))


@torch.no_grad()
def cache_teacher_outputs(model, loader, device):
    model.eval()
    all_logits, all_latents = [], []
    for batch in loader:
        logits, latent = model(move_batch(batch, device), return_latent=True)
        all_logits.append(logits.cpu())
        all_latents.append(latent.cpu())
    return torch.cat(all_logits), torch.cat(all_latents)

## Training loops

Both use AdamW + cosine LR decay + mixed precision (AMP), and early-stop on validation MPR with
`PATIENCE` epochs of no improvement - so you're not locked into the full epoch count if the model
converges early.

In [8]:
def train_teacher(model, train_loader, val_loader, epochs, device, out_path):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    focal = FocalLoss(gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_mpr, best_state, patience_left = -1.0, None, PATIENCE
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            labels = batch["label"].to(device)
            batch_dev = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(batch_dev)
                loss = focal(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
        scheduler.step()

        val_metrics = evaluate(model, val_loader, device)
        print(f"[Teacher] epoch {epoch+1}/{epochs}  "
              f"train_loss={running_loss/len(train_loader.dataset):.4f}  "
              f"val_top1={val_metrics['top1']:.4f}  val_mpr={val_metrics['mpr']:.4f}  "
              f"({time.time()-t0:.1f}s)")

        if val_metrics["mpr"] > best_mpr:
            best_mpr, best_state, patience_left = val_metrics["mpr"], \
                {k: v.cpu().clone() for k, v in model.state_dict().items()}, PATIENCE
            torch.save(best_state, out_path)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch+1} (no val MPR gain in {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)
    return model


def train_student_kd(student, train_loader, val_loader, teacher_logits, teacher_latents,
                      epochs, device, out_path):
    student.to(device)
    teacher_logits = teacher_logits.to(device)
    teacher_latents = teacher_latents.to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    focal = FocalLoss(gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_mpr, best_state, patience_left = -1.0, None, PATIENCE
    for epoch in range(epochs):
        t0 = time.time()
        student.train()
        running_loss = 0.0
        for batch in train_loader:
            labels = batch["label"].to(device)
            idx = batch["idx"].to(device)
            batch_dev = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                s_logits, s_latent = student(batch_dev, return_latent=True)
                t_logits, t_latent = teacher_logits[idx], teacher_latents[idx]

                loss_focal = focal(s_logits, labels)
                loss_kd = kd_kl_loss(s_logits, t_logits)
                loss_extra = relational_loss(s_latent, t_latent) if USE_RELATIONAL \
                    else F.mse_loss(s_latent, t_latent)
                loss = (1 - KD_ALPHA) * loss_focal + KD_ALPHA * (loss_kd + loss_extra)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
        scheduler.step()

        val_metrics = evaluate(student, val_loader, device)
        print(f"[Student-KD] epoch {epoch+1}/{epochs}  "
              f"train_loss={running_loss/len(train_loader.dataset):.4f}  "
              f"val_top1={val_metrics['top1']:.4f}  val_mpr={val_metrics['mpr']:.4f}  "
              f"({time.time()-t0:.1f}s)")

        if val_metrics["mpr"] > best_mpr:
            best_mpr, best_state, patience_left = val_metrics["mpr"], \
                {k: v.cpu().clone() for k, v in student.state_dict().items()}, PATIENCE
            torch.save(best_state, out_path)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch+1}")
                break

    student.load_state_dict(best_state)
    return student


def train_student_plain(student, train_loader, val_loader, epochs, device, out_path):
    '''WithoutKD baseline: same radar-only architecture, plain focal loss only.'''
    student.to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    focal = FocalLoss(gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_mpr, best_state, patience_left = -1.0, None, PATIENCE
    for epoch in range(epochs):
        t0 = time.time()
        student.train()
        running_loss = 0.0
        for batch in train_loader:
            labels = batch["label"].to(device)
            batch_dev = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = student(batch_dev)
                loss = focal(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
        scheduler.step()

        val_metrics = evaluate(student, val_loader, device)
        print(f"[WithoutKD] epoch {epoch+1}/{epochs}  "
              f"train_loss={running_loss/len(train_loader.dataset):.4f}  "
              f"val_top1={val_metrics['top1']:.4f}  val_mpr={val_metrics['mpr']:.4f}  "
              f"({time.time()-t0:.1f}s)")

        if val_metrics["mpr"] > best_mpr:
            best_mpr, best_state, patience_left = val_metrics["mpr"], \
                {k: v.cpu().clone() for k, v in student.state_dict().items()}, PATIENCE
            torch.save(best_state, out_path)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch+1}")
                break

    student.load_state_dict(best_state)
    return student

## Run everything

Order: Teacher -> cache Teacher outputs on the train set -> KD student -> WithoutKD baseline ->
final comparison table. Checkpoints for all three models are saved to `/kaggle/working/` so you can
resume or reload without retraining if the session gets interrupted.

In [9]:
overall_t0 = time.time()

print("=" * 70)
print("STAGE 1/4: Training Teacher (multimodal)")
print("=" * 70)
teacher = TeacherModel()
teacher = train_teacher(teacher, train_loader, val_loader, TEACHER_EPOCHS, DEVICE,
                         os.path.join(OUT_DIR, "teacher_best.pt"))

print("\n" + "=" * 70)
print("STAGE 2/4: Caching Teacher outputs on the train set")
print("=" * 70)
teacher_logits, teacher_latents = cache_teacher_outputs(teacher, train_loader_ordered, DEVICE)
print("Cached teacher logits/latents:", teacher_logits.shape, teacher_latents.shape)

print("\n" + "=" * 70)
print("STAGE 3/4: Training Student with KD (radar-only)")
print("=" * 70)
student_kd = StudentModel()
student_kd = train_student_kd(student_kd, train_loader, val_loader, teacher_logits, teacher_latents,
                               STUDENT_EPOCHS, DEVICE, os.path.join(OUT_DIR, "student_kd_best.pt"))

print("\n" + "=" * 70)
print("STAGE 4/4: Training WithoutKD baseline (radar-only, no distillation)")
print("=" * 70)
student_plain = StudentModel()
student_plain = train_student_plain(student_plain, train_loader, val_loader, STUDENT_EPOCHS, DEVICE,
                                     os.path.join(OUT_DIR, "student_withoutkd_best.pt"))

print("\n" + "=" * 70)
print("FINAL COMPARISON ON VALIDATION SET")
print("=" * 70)
results = {
    "Teacher (multimodal)": evaluate(teacher, val_loader, DEVICE),
    "Student - WithoutKD (radar only)": evaluate(student_plain, val_loader, DEVICE),
    "Student - KD (radar only)": evaluate(student_kd, val_loader, DEVICE),
}
for name, m in results.items():
    print(f"{name:35s}  top1={m['top1']:.4f}  top5={m['top5']:.4f}  top10={m['top10']:.4f}  "
          f"mean_rss={m['mean_rss']:.2f} dBm  mpr={m['mpr']:.4f}")

elapsed_hr = (time.time() - overall_t0) / 3600
print(f"\nTotal wall-clock time: {elapsed_hr:.2f} hours")
print("Checkpoints saved under /kaggle/working/ (teacher_best.pt, student_kd_best.pt, student_withoutkd_best.pt)")

STAGE 1/4: Training Teacher (multimodal)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 234MB/s]
/tmp/ipykernel_58/1979911228.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_58/1979911228.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[Teacher] epoch 1/25  train_loss=2.8369  val_top1=0.1933  val_mpr=0.7866  (34.2s)
[Teacher] epoch 2/25  train_loss=1.8201  val_top1=0.2667  val_mpr=0.8597  (16.2s)
[Teacher] epoch 3/25  train_loss=1.4854  val_top1=0.2600  val_mpr=0.8770  (15.7s)
[Teacher] epoch 4/25  train_loss=1.3257  val_top1=0.2830  val_mpr=0.8942  (13.8s)
[Teacher] epoch 5/25  train_loss=1.1965  val_top1=0.3281  val_mpr=0.9140  (13.6s)
[Teacher] epoch 6/25  train_loss=1.1167  val_top1=0.3163  val_mpr=0.9174  (13.5s)
[Teacher] epoch 7/25  train_loss=1.0185  val_top1=0.3148  val_mpr=0.9237  (13.3s)
[Teacher] epoch 8/25  train_loss=0.9502  val_top1=0.3081  val_mpr=0.9278  (13.5s)
[Teacher] epoch 9/25  train_loss=0.8913  val_top1=0.3096  val_mpr=0.9078  (13.5s)
[Teacher] epoch 10/25  train_loss=0.8091  val_top1=0.2970  val_mpr=0.9177  (13.3s)
[Teacher] epoch 11/25  train_loss=0.7469  val_top1=0.2970  val_mpr=0.9250  (13.6s)
[Teacher] epoch 12/25  train_loss=0.6781  val_top1=0.2867  val_mpr=0.9312  (13.4s)
[Teacher] epo

/tmp/ipykernel_58/1979911228.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_58/1979911228.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[Student-KD] epoch 1/25  train_loss=4.2148  val_top1=0.1185  val_mpr=0.6812  (12.9s)
[Student-KD] epoch 2/25  train_loss=3.8306  val_top1=0.1126  val_mpr=0.6669  (13.3s)
[Student-KD] epoch 3/25  train_loss=3.6977  val_top1=0.1044  val_mpr=0.6522  (13.6s)
[Student-KD] epoch 4/25  train_loss=3.5135  val_top1=0.1089  val_mpr=0.6824  (13.5s)
[Student-KD] epoch 5/25  train_loss=3.2522  val_top1=0.1059  val_mpr=0.6846  (13.4s)
[Student-KD] epoch 6/25  train_loss=2.9731  val_top1=0.1007  val_mpr=0.6962  (13.1s)
[Student-KD] epoch 7/25  train_loss=2.6662  val_top1=0.0978  val_mpr=0.7062  (13.0s)
[Student-KD] epoch 8/25  train_loss=2.5032  val_top1=0.1015  val_mpr=0.6792  (12.9s)
[Student-KD] epoch 9/25  train_loss=2.2951  val_top1=0.1104  val_mpr=0.7344  (12.8s)
[Student-KD] epoch 10/25  train_loss=2.1109  val_top1=0.0689  val_mpr=0.7081  (13.0s)
[Student-KD] epoch 11/25  train_loss=1.9507  val_top1=0.0948  val_mpr=0.7189  (12.8s)
[Student-KD] epoch 12/25  train_loss=1.7962  val_top1=0.1096  v

/tmp/ipykernel_58/1979911228.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_58/1979911228.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[WithoutKD] epoch 1/25  train_loss=3.3965  val_top1=0.1185  val_mpr=0.6773  (12.8s)
[WithoutKD] epoch 2/25  train_loss=3.1274  val_top1=0.1170  val_mpr=0.6763  (12.8s)
[WithoutKD] epoch 3/25  train_loss=3.0341  val_top1=0.1111  val_mpr=0.6621  (12.7s)
[WithoutKD] epoch 4/25  train_loss=2.9436  val_top1=0.0911  val_mpr=0.6572  (12.7s)
[WithoutKD] epoch 5/25  train_loss=2.8573  val_top1=0.0652  val_mpr=0.6410  (12.8s)
[WithoutKD] epoch 6/25  train_loss=2.7522  val_top1=0.0548  val_mpr=0.6661  (12.6s)
[WithoutKD] epoch 7/25  train_loss=2.6212  val_top1=0.0896  val_mpr=0.6784  (12.7s)
[WithoutKD] epoch 8/25  train_loss=2.4079  val_top1=0.0904  val_mpr=0.6828  (12.5s)
[WithoutKD] epoch 9/25  train_loss=2.2747  val_top1=0.1067  val_mpr=0.7075  (12.7s)
[WithoutKD] epoch 10/25  train_loss=2.1293  val_top1=0.0800  val_mpr=0.7006  (12.9s)
[WithoutKD] epoch 11/25  train_loss=2.0075  val_top1=0.1089  val_mpr=0.7076  (12.9s)
[WithoutKD] epoch 12/25  train_loss=1.8721  val_top1=0.1244  val_mpr=0.722

## If you have GPU-hour budget left over

- Raise `IMG_SIZE` back toward 256 (the paper's setting) - the modest amount of full-run headroom this
  smaller dataset gives you (vs. the ~6-10 hour estimate) is a reasonable place to spend it.
- Raise `TEACHER_EPOCHS` / `STUDENT_EPOCHS` and let early stopping decide when to actually stop.
- Use all 5 frames (`image_1..5`, `lidar_bev_1..5`, `radar_1..5`) instead of just the latest one - this
  needs a small architecture change (e.g. a 3D-conv or per-frame-then-pool encoder) but would better
  match the paper's temporal-window setup.
- Re-run Stage 3 with `USE_RELATIONAL = False` to get a plain-KD student for a 3-way comparison against
  the relational-KD student, mirroring the paper's `KD` vs. `RKD` rows in Table II.